<a href="https://colab.research.google.com/github/KaabiHiba/FER-CE-Project/blob/main/Hiba_Moncef_version0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = "/content/drive/MyDrive/compound"
IMAGE_PATH = os.path.join(BASE_PATH, "Image/aligned")
LABEL_PATH = os.path.join(BASE_PATH, "EmoLabel/list_patition_label.txt")

labels_dict = {}

with open(LABEL_PATH, "r") as f:
    for line in f:
        name, label = line.strip().split()
        labels_dict[name] = int(label) - 1  # 🔥 start from 0

In [ ]:
train_images = []
test_images = []

for name in labels_dict.keys():
    if "train" in name:
        train_images.append(name)
    else:
        test_images.append(name)

print(len(train_images), len(test_images))

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        real_name = img_name.replace(".jpg", "_aligned.jpg")
        img_path = os.path.join(IMAGE_PATH, real_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Return black image + correct label to avoid crashing
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        label = labels_dict[img_name]
        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
train_data, val_data = train_test_split(train_images, test_size=0.2, random_state=42)

train_dataset = EmotionDataset(train_data, transform=train_transform)
val_dataset = EmotionDataset(val_data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, drop_last=True, num_workers=2)

In [ ]:
labels_array = [labels_dict[x] for x in train_data]

num_classes = 14
class_weights = np.ones(num_classes)

unique_classes = np.unique(labels_array)

computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=labels_array
)

for i, c in enumerate(unique_classes):
    class_weights[c] = computed_weights[i]

weights = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

In [ ]:
model = models.resnet18(weights="DEFAULT")

for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.6),
    nn.Linear(512, 14)
)

model = model.to(device)

In [ ]:
weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=2,
    factor=0.3
)

In [ ]:
num_epochs = 25
best_acc = 0

for epoch in range(num_epochs):
    print(f"\n🚀 Epoch {epoch+1}/{num_epochs}")

    model.train()
    running_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 20 == 0:
            print(f"Batch {i}/{len(train_loader)} - Loss: {loss.item():.4f}")

    # VALIDATION
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    acc = 100 * correct / total
    epoch_loss = running_loss / len(train_loader)

    print(f"✅ Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Acc: {acc:.2f}%")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_model.pth")
        print("🔥 Best model saved!")

In [ ]:
class_names = [
    "Neutral", "Happy", "Sad", "Surprise",
    "Fear", "Disgust", "Angry", "Contempt",
    "Happily Surprised", "Sadly Angry",
    "Sadly Surprised", "Fearfully Surprised",
    "Angrily Surprised", "Disgustedly Surprised"
]

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pth"))
model.eval()

import matplotlib.pyplot as plt
import random

for i in range(5):
    img, label = val_dataset[random.randint(0, len(val_dataset)-1)]

    input_img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_img)
        _, pred = torch.max(output, 1)

    img_show = img.permute(1,2,0).cpu().numpy()
    img_show = (img_show * 0.5) + 0.5

    plt.imshow(img_show)
    plt.title(f"True: {class_names[label]} | Pred: {class_names[pred.item()]}")
    plt.axis("off")
    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Report
print(classification_report(y_true, y_pred))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

print("✅ BLIP model loaded")

In [ ]:
from PIL import Image

def generate_caption(image_path):
    image = Image.open(image_path).convert("RGB")

    prompt = "Describe the facial emotion and explain the facial features (eyes, mouth, eyebrows)."

    inputs = processor(image, text=prompt, return_tensors="pt").to(device)

    out = blip_model.generate(**inputs, max_new_tokens=50)

    caption = processor.decode(out[0], skip_special_tokens=True)

    return caption

In [ ]:
emotion_explanations = {
    "happily surprised": "smiling mouth and raised eyebrows",
    "sadly angry": "frowning eyebrows and tense lips",
    "fearfully disgusted": "wide eyes and wrinkled nose",
    "happily disgusted": "smile with slight nose wrinkle",
    "sadly fearful": "drooping eyes and slightly open mouth",
    "angrily surprised": "wide eyes with tense eyebrows",
    "sadly surprised": "raised eyebrows with sad mouth",
    "disgustedly surprised": "wrinkled nose and raised eyebrows",
    "fearfully surprised": "wide eyes and open mouth",
    "angrily disgusted": "tight lips and wrinkled nose",
    "sadly disgusted": "downturned lips and nose wrinkle",
    "happily angry": "smile with tense eyebrows",
    "fearfully angry": "wide eyes with tight lips",
    "neutral": "relaxed face with no strong emotion"
}

In [ ]:
import matplotlib.pyplot as plt

def predict_with_explanation(img_path):

    image = Image.open(img_path).convert("RGB")

    plt.imshow(image)
    plt.title("Image")
    plt.axis("off")
    plt.show()

    img = val_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img)
        _, pred = torch.max(output, 1)

    emotion = emotion_labels[pred.item()]
    explanation = emotion_explanations[emotion]

    print("😊 Emotion prédite :", emotion)
    print("🧠 Explication :", explanation)

In [ ]:
emotion_labels = [
    "happily surprised", "sadly angry", "fearfully disgusted",
    "happily disgusted", "sadly fearful", "angrily surprised",
    "sadly surprised", "disgustedly surprised",
    "fearfully surprised", "angrily disgusted",
    "sadly disgusted", "happily angry",
    "fearfully angry", "neutral"
]

import matplotlib.pyplot as plt

def predict_with_explanation(img_path):

    # 🔹 afficher image
    image = Image.open(img_path).convert("RGB")
    plt.imshow(image)
    plt.title("Image test")
    plt.axis("off")
    plt.show()

    # 🔹 CNN
    img = val_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img)
        _, pred = torch.max(output, 1)

    emotion = emotion_labels[pred.item()]

    # 🔹 LLM
    caption = generate_caption(img_path)

    print("😊 Emotion prédite :", emotion)
    print("🧠 Explication :", caption)

In [ ]:
test_img = "/content/drive/MyDrive/compound/Image/aligned/train_0005_aligned.jpg"

predict_with_explanation(test_img)

In [ ]:
!pip install open_clip_torch

In [ ]:
import torch
import open_clip
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_clip, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
model_clip = model_clip.to(device)

In [ ]:
text_prompts = [
    "a person smiling with raised eyebrows",
    "a person with angry eyebrows and sad mouth",
    "a person with wide eyes and disgusted face",
    "a smiling person with slight disgust",
    "a sad person with fearful eyes",
    "an angry person with surprised expression",
    "a sad person with raised eyebrows",
    "a disgusted person with surprised face",
    "a person with wide eyes and open mouth",
    "an angry person with disgust expression",
    "a sad person with disgust expression",
    "a smiling person with anger",
    "a fearful person with anger",
    "a neutral face"
]

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# ✅ dictionnaire d'explications
emotion_explanations = {
    "a person smiling with raised eyebrows": "Positive emotion with engagement (joy + surprise)",
    "a person with angry eyebrows and sad mouth": "Mixed negative emotion (anger + sadness)",
    "a person with wide eyes and disgusted face": "Disgust combined with surprise",
    "a smiling person with slight disgust": "Conflicting signals (joy + disgust)",
    "a sad person with fearful eyes": "Fear mixed with sadness",
    "an angry person with surprised expression": "High arousal emotion (anger + surprise)",
    "a sad person with raised eyebrows": "Sadness with concern or worry",
    "a disgusted person with surprised face": "Disgust mixed with surprise",
    "a person with wide eyes and open mouth": "Strong surprise or fear",
    "an angry person with disgust expression": "Intense negative emotion (anger + disgust)",
    "a sad person with disgust expression": "Negative emotional blend",
    "a smiling person with anger": "Mixed emotion (joy + anger)",
    "a fearful person with anger": "Fear combined with aggression",
    "a neutral face": "No strong emotion detected"
}

def predict_clip_full(image_path):

    # 🔹 afficher image
    image = Image.open(image_path).convert("RGB")
    plt.imshow(image)
    plt.title("Image test")
    plt.axis("off")
    plt.show()

    # 🔹 preprocess
    image_input = preprocess(image).unsqueeze(0).to(device)
    text = open_clip.tokenize(text_prompts).to(device)

    with torch.no_grad():
        image_features = model_clip.encode_image(image_input)
        text_features = model_clip.encode_text(text)

        # 🔥 normalisation (très important)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        logits = image_features @ text_features.T
        probs = logits.softmax(dim=-1)

    pred = probs.argmax().item()

    # 🔹 émotion détectée
    emotion = text_prompts[pred]

    # 🔹 explication
    explanation = emotion_explanations.get(
        emotion,
        "Facial expression detected but not clearly classified"
    )

    # 🔹 affichage
    print("😊 Emotion (CLIP) :", emotion)
    print("🧠 Explication :", explanation)
    print("📊 Probabilités :", probs.cpu().numpy())

In [ ]:
img_path = "/content/drive/MyDrive/compound/Image/aligned/train_0005_aligned.jpg"
predict_clip_full(img_path)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image, class_idx):
        self.model.zero_grad()

        output = self.model(input_image)
        loss = output[0, class_idx]
        loss.backward()

        gradients = self.gradients[0].cpu().data.numpy()
        activations = self.activations[0].cpu().data.numpy()

        weights = np.mean(gradients, axis=(1, 2))

        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam - cam.min()
        cam = cam / cam.max()

        return cam

In [ ]:
target_layer = model.layer4
gradcam = GradCAM(model, target_layer)

In [ ]:
def show_gradcam(img_path):
    model.eval()

    img = Image.open(img_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    input_tensor = transform(img).unsqueeze(0).to(device)

    # prediction
    output = model(input_tensor)
    _, pred = torch.max(output, 1)

    # Grad-CAM
    cam = gradcam.generate(input_tensor, pred.item())

    # image originale
    img_np = np.array(img.resize((224,224)))

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255

    superimposed = heatmap + np.float32(img_np)/255
    superimposed = superimposed / np.max(superimposed)

    # affichage
    plt.figure(figsize=(10,4))

    plt.subplot(1,3,1)
    plt.imshow(img_np)
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(cam, cmap='jet')
    plt.title("Heatmap")
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(superimposed)
    plt.title(f"Pred: {pred.item()}")
    plt.axis('off')

    plt.show()

In [ ]:
img_path = "/content/drive/MyDrive/compound/Image/aligned/train_0007_aligned.jpg"

show_gradcam(img_path)

In [ ]:
def compare_predictions(img_path):
    model.eval()

    img = Image.open(img_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    input_tensor = transform(img).unsqueeze(0).to(device)

    # prediction
    output = model(input_tensor)
    _, pred = torch.max(output, 1)

    print("Prediction ResNet:", pred.item())

    # Grad-CAM
    cam = gradcam.generate(input_tensor, pred.item())

    plt.imshow(cam, cmap='jet')
    plt.title("Zones importantes (Grad-CAM)")
    plt.axis('off')
    plt.show()

In [ ]:
compare_predictions("/content/drive/MyDrive/compound/Image/aligned/train_0009_aligned.jpg")

In [ ]:
def analyze_attention(cam):
    h, w = cam.shape

    top = cam[:h//3, :]
    middle = cam[h//3:2*h//3, :]
    bottom = cam[2*h//3:, :]

    top_score = np.mean(top)
    mid_score = np.mean(middle)
    bot_score = np.mean(bottom)

    if top_score > mid_score and top_score > bot_score:
        return "eyes"
    elif bot_score > mid_score and bot_score > top_score:
        return "mouth"
    else:
        return "center_face"

In [ ]:
def generate_explanation(pred_class, focus_area):

    emotions = [
        "Neutral", "Happy", "Sad", "Surprise", "Fear",
        "Disgust", "Anger", "Contempt", "Anxiety",
        "Helplessness", "Disappointment", "Boredom",
        "Calm", "Confusion"
    ]

    emotion = emotions[pred_class]

    if focus_area == "eyes":
        focus_text = "the eyes"
    elif focus_area == "mouth":
        focus_text = "the mouth"
    else:
        focus_text = "the central facial region"

    explanation = f"The model predicts '{emotion}' because it focuses mainly on {focus_text}."

    return explanation

In [ ]:
def show_gradcam_with_text(img_path):
    model.eval()

    img = Image.open(img_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    input_tensor = transform(img).unsqueeze(0).to(device)

    # prediction
    output = model(input_tensor)
    pred_class = output.argmax(dim=1).item()

    # Grad-CAM
    cam = gradcam.generate(input_tensor, pred_class)

    # 🔥 analyse zones
    focus_area = analyze_attention(cam)

    # 🔥 génération texte
    explanation = generate_explanation(pred_class, focus_area)

    # image originale
    img_np = np.array(img.resize((224,224)))

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255

    superimposed = heatmap + np.float32(img_np)/255
    superimposed = superimposed / np.max(superimposed)

    # affichage
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(img_np)
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(cam, cmap='jet')
    plt.title("Heatmap")
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(superimposed)
    plt.title(f"Pred: {pred_class}")
    plt.axis('off')

    plt.show()

    # 🔥 texte final
    print("🧠 Explanation:")
    print(explanation)

In [ ]:
show_gradcam_with_text(img_path)

In [ ]:
def full_test(img_path):
    model.eval()

    # preprocess
    img = Image.open(img_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    input_tensor = transform(img).unsqueeze(0).to(device)

    # prediction
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)
    top3_prob, top3_class = torch.topk(probs, 3)

    pred_class = top3_class[0][0].item()

    print("Top 3 classes:", top3_class)
    print("Probabilities:", top3_prob)

    # Grad-CAM
    cam = gradcam.generate(input_tensor, pred_class)

    # affichage
    img_np = np.array(img.resize((224,224)))

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255

    superimposed = heatmap + np.float32(img_np)/255
    superimposed = superimposed / np.max(superimposed)

    plt.figure(figsize=(10,4))

    plt.subplot(1,3,1)
    plt.imshow(img_np)
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(cam, cmap='jet')
    plt.title("Grad-CAM")
    plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(superimposed)
    plt.title(f"Pred: {pred_class}")
    plt.axis('off')

    plt.show()

    return pred_class

In [ ]:
for i in range(1, 6):
    path = f"/content/drive/MyDrive/compound/Image/aligned/train_000{i}_aligned.jpg"
    print("IMAGE", i)
    full_test(path)

In [ ]:
def explain_prediction(pred_class):
    emotions = [
        "Happiness", "Sadness", "Surprise", "Fear",
        "Anger", "Disgust", "Contempt", "Neutral",
        "Anxiety", "Helplessness", "Disappointment",
        "Pride", "Shame", "Interest"
    ]

    return f"The model predicts '{emotions[pred_class]}' because it focuses on key facial regions like eyes and mouth."

In [ ]:
img_path_to_explain = '/content/drive/MyDrive/compound/Image/aligned/train_0009_aligned.jpg'
pred_class_for_explanation = full_test(img_path_to_explain)
print(explain_prediction(pred_class_for_explanation))

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from PIL import Image
import torch
from torchvision import transforms

# Labels (adapter selon ton dataset)
emotion_labels = [
    "Neutral", "Happy", "Sad", "Surprise", "Fear",
    "Disgust", "Anger", "Contempt", "Anxiety",
    "Helplessness", "Disappointment", "Other1", "Other2", "Other3"
]

def predict_and_explain(image):
    # Transform
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    img = transform(image).unsqueeze(0).to(device)

    # Prediction
    model.eval()
    with torch.no_grad():
        output = model(img)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    # Texte explicatif (Vision-LLM simple)
    explanation = f"The model predicts '{emotion_labels[pred]}' because it focuses on key facial regions."

    return emotion_labels[pred], explanation

In [ ]:
interface = gr.Interface(
    fn=predict_and_explain,
    inputs=gr.Image(type="pil"),
    outputs=["label", "text"],
    title="FER Emotion Recognition",
    description="Upload an image to detect emotion"
)

interface.launch()